# Feature Selection Using Filter Methods — data-2.csv

This notebook applies filter-based feature selection to `data-2.csv`.

The following methods are demonstrated:
1. Variance Threshold
2. Correlation Coefficient
3. Chi-Square Test
4. ANOVA F-Test
5. Mutual Information

For this dataset, `Promoted` is used as the target variable and the remaining columns are treated as input features.

> **Note:** This dataset contains only 6 observations, so the statistical scores and p-values should be treated as a demonstration of the methods rather than strong statistical evidence.


In [ ]:
# Import the required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    chi2,
    f_classif,
    mutual_info_classif
)

from sklearn.preprocessing import MinMaxScaler


## 1. Load the Dataset

The CSV file is loaded into a pandas DataFrame.

We inspect:
- Dataset shape
- First few records
- Data types
- Missing values


In [ ]:
# Load the dataset
df = pd.read_csv("D:\BDA_0024\AML_LAB\Datasets\data-2.csv")

print("Dataset Shape:", df.shape)

print("\nFirst 5 rows:")
display(df.head())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())


## 2. Separate Features and Target

Here, `Promoted` is considered the target variable.

- `X` = input features
- `y` = target variable

If your actual target is a different column, change `target_column` accordingly.


In [ ]:
# Define the target column
target_column = "Promoted"

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column]

print("Input Features:")
print(X.columns.tolist())

print("\nTarget Variable:")
print(target_column)

print("\nTarget Class Distribution:")
print(y.value_counts())


## 3. Descriptive Statistics

`describe()` gives basic statistical information about the numerical features, including count, mean, standard deviation, minimum, maximum, and quartiles.


In [ ]:
# Display descriptive statistics
display(df.describe())


## 4. Filter Method 1 — Variance Threshold

Variance Threshold removes features with little or no variation.

A feature with zero variance has the same value for every observation and cannot help distinguish between observations.

Here, `threshold=0.0` removes only constant features.


In [ ]:
# Create the Variance Threshold selector
variance_selector = VarianceThreshold(threshold=0.0)

# Fit the selector
variance_selector.fit(X)

# Get the Boolean mask of selected features
selected_mask = variance_selector.get_support()

# Get selected and removed feature names
variance_selected = X.columns[selected_mask].tolist()
variance_removed = X.columns[~selected_mask].tolist()

print("Features selected by Variance Threshold:")
print(variance_selected)

print("\nFeatures removed:")
print(variance_removed)


In [ ]:
# Calculate feature variance
feature_variance = X.var()

print("Variance of each feature:")
display(feature_variance.to_frame(name="Variance"))

# Visualize feature variance
plt.figure(figsize=(8, 5))
feature_variance.plot(kind="bar")

plt.title("Variance of Input Features")
plt.xlabel("Features")
plt.ylabel("Variance")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


## 5. Filter Method 2 — Correlation Coefficient

Correlation measures the strength and direction of a linear relationship between numerical variables.

Values close to:
- `+1` indicate a strong positive linear relationship
- `-1` indicate a strong negative linear relationship
- `0` indicate little or no linear relationship

Highly correlated features may contain redundant information.


In [ ]:
# Calculate the correlation matrix for input features
correlation_matrix = X.corr()

print("Feature Correlation Matrix:")
display(correlation_matrix)

# Visualize the correlation matrix
plt.figure(figsize=(8, 6))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.title("Correlation Matrix of Input Features")
plt.tight_layout()
plt.show()


In [ ]:
# Identify highly correlated features
# 0.80 is used as an example threshold

feature_corr = X.corr().abs()

upper_triangle = feature_corr.where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)

high_correlation = [
    column
    for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.80)
]

print("Features involved in correlation greater than 0.80:")
print(high_correlation)


## 6. Filter Method 3 — Chi-Square Test

The Chi-Square test evaluates the association between each feature and a categorical target.

The `chi2` implementation requires non-negative feature values. Therefore, the features are scaled to the range `[0, 1]` before applying the test.

A larger Chi-Square score indicates a stronger observed association with the target, while the p-value is used to assess statistical significance.

Because this dataset has only six observations, the results should be interpreted cautiously.


In [ ]:
# Check whether the target contains at least two classes
print("Number of target classes:", y.nunique())

if y.nunique() >= 2:
    # Convert feature values to a non-negative range
    scaler = MinMaxScaler()
    X_non_negative = scaler.fit_transform(X)

    # Apply Chi-Square feature selection
    chi_selector = SelectKBest(
        score_func=chi2,
        k="all"
    )

    chi_selector.fit(X_non_negative, y)

    # Store Chi-Square results
    chi2_results = pd.DataFrame({
        "Feature": X.columns,
        "Chi2 Score": chi_selector.scores_,
        "P-Value": chi_selector.pvalues_
    })

    # Sort from highest to lowest Chi-Square score
    chi2_results = chi2_results.sort_values(
        by="Chi2 Score",
        ascending=False
    )

    display(chi2_results)

else:
    print("Chi-Square cannot be applied because the target has fewer than two classes.")


## 7. Filter Method 4 — ANOVA F-Test

The ANOVA F-test evaluates whether the mean value of a numerical feature differs across the classes of a categorical target.

The F-statistic compares:
- Variation between target groups
- Variation within target groups

A larger F-score indicates stronger separation between target groups.

The p-value can be used to assess statistical significance.


In [ ]:
# Apply ANOVA F-test
if y.nunique() >= 2:
    anova_selector = SelectKBest(
        score_func=f_classif,
        k="all"
    )

    anova_selector.fit(X, y)

    # Store ANOVA results
    anova_results = pd.DataFrame({
        "Feature": X.columns,
        "ANOVA F-Score": anova_selector.scores_,
        "P-Value": anova_selector.pvalues_
    })

    # Sort by F-score
    anova_results = anova_results.sort_values(
        by="ANOVA F-Score",
        ascending=False
    )

    display(anova_results)

else:
    print("ANOVA cannot be applied because the target has fewer than two classes.")


## 8. Filter Method 5 — Mutual Information

Mutual Information measures how much information a feature provides about the target.

Unlike simple linear correlation, Mutual Information can detect more general dependencies between a feature and target.

A higher score indicates greater estimated dependency.

Since the dataset is very small, Mutual Information scores may vary considerably and should not be treated as definitive.


In [ ]:
# Apply Mutual Information
if y.nunique() >= 2:
    mi_scores = mutual_info_classif(
        X,
        y,
        random_state=42
    )

    # Create result table
    mi_results = pd.DataFrame({
        "Feature": X.columns,
        "Mutual Information": mi_scores
    })

    # Sort from highest to lowest score
    mi_results = mi_results.sort_values(
        by="Mutual Information",
        ascending=False
    )

    display(mi_results)

else:
    print("Mutual Information cannot be applied because the target has fewer than two classes.")


In [ ]:
# Visualize Mutual Information scores

if y.nunique() >= 2:
    plt.figure(figsize=(8, 5))

    plt.bar(
        mi_results["Feature"],
        mi_results["Mutual Information"]
    )

    plt.title("Mutual Information Feature Scores")
    plt.xlabel("Features")
    plt.ylabel("Mutual Information")

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 9. Compare the Filter Methods

The following section combines the feature scores from the different filter methods.

The raw scores should **not** be added directly because each method uses a different scoring scale. Instead, the individual rankings can be compared.


In [ ]:
# Create a comparison table

comparison = pd.DataFrame({
    "Feature": X.columns,
    "Variance": X.var().values
})

# Add Chi-Square scores if available
if y.nunique() >= 2:
    comparison["Chi2 Score"] = chi_selector.scores_

# Add ANOVA scores if available
if y.nunique() >= 2:
    comparison["ANOVA F-Score"] = anova_selector.scores_

# Add Mutual Information scores if available
if y.nunique() >= 2:
    comparison["Mutual Information"] = mi_scores

display(comparison)


## 10. Rank Features

Because the different filter methods produce scores on different scales, ranking each method separately is more appropriate than directly combining the raw scores.


In [ ]:
# Create ranking columns

comparison["Variance Rank"] = (
    comparison["Variance"]
    .rank(ascending=False, method="min")
)

if "Chi2 Score" in comparison.columns:
    comparison["Chi2 Rank"] = (
        comparison["Chi2 Score"]
        .rank(ascending=False, method="min")
    )

if "ANOVA F-Score" in comparison.columns:
    comparison["ANOVA Rank"] = (
        comparison["ANOVA F-Score"]
        .rank(ascending=False, method="min")
    )

if "Mutual Information" in comparison.columns:
    comparison["MI Rank"] = (
        comparison["Mutual Information"]
        .rank(ascending=False, method="min")
    )

display(comparison)


## 11. Final Feature Selection

For this notebook, the final dataset is created using the features that passed the Variance Threshold.

For a real machine-learning project, the final feature set should be decided using the results of several appropriate filter methods and then validated using a model on unseen data.


In [ ]:
# Select features that passed the variance filter
X_selected = X[variance_selected]

print("Final selected features:")
print(X_selected.columns.tolist())

print("\nFinal selected feature dataset:")
display(X_selected)


## 12. Summary

### Filter Methods

| Method | Purpose |
|---|---|
| Variance Threshold | Removes constant or low-variance features |
| Correlation | Identifies redundant highly correlated features |
| Chi-Square | Measures association between features and a categorical target |
| ANOVA F-Test | Measures differences in feature means between target classes |
| Mutual Information | Measures dependency/information shared between a feature and target |

### Dataset-Specific Observation

`data-2.csv` contains 6 observations and 5 columns. `Promoted` is used as the target, giving two target classes (`0` and `1`), so the classification-based filter methods can technically be applied.

However, because there are only six observations, the resulting scores and p-values are not sufficient to make strong statistical conclusions. A larger dataset should be used for reliable feature selection.
